In [7]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC

import pickle
from geoai.utils_geo.RasterOps import RasterOperations
from geoai.utils_ml.ModelOps import ModelOperations

raster_ops = RasterOperations()
model_ops = ModelOperations()

In [8]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train.csv")
y_test = pd.read_csv("csv_files/y_test.csv")

# compute indices
X_train = raster_ops.compute_ndvi_using_df(X_train, "NIR", "RED")
X_train = raster_ops.compute_ndbi(X_train, "NIR", "SWIR")
X_train = raster_ops.compute_rei(X_train, "NIR", "BLUE")
X_test = raster_ops.compute_ndvi_using_df(X_test, "NIR", "RED")
X_test = raster_ops.compute_ndbi(X_test, "NIR", "SWIR")
X_test = raster_ops.compute_rei(X_test, "NIR", "BLUE")


# create binary and discrete NDVI
X_train = raster_ops.create_ndvi_bin(X_train, "NDVI")
X_train = raster_ops.create_ndvi_discrete(X_train, "NDVI")
X_test = raster_ops.create_ndvi_bin(X_test, "NDVI")
X_test = raster_ops.create_ndvi_discrete(X_test, "NDVI")

In [9]:
custom_class_weights = {0: 1, 1: 1, 2: 10, 3: 1} 
cat = CatBoostClassifier(loss_function='MultiClass', learning_rate=0.05, verbose=0, class_weights=custom_class_weights)
lr = LogisticRegression(C=9.232675920286502, max_iter=300, solver="lbfgs", class_weight=custom_class_weights)
rf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight=custom_class_weights)
svc = SVC(class_weight=custom_class_weights)

In [10]:
pipeline_1 = model_ops.make_pipeline(cat)
pipeline_2 = model_ops.make_pipeline(lr)
pipeline_3 = model_ops.make_pipeline(rf)
pipeline_4 = model_ops.make_pipeline(svc)

estimators = [
    ("cat", pipeline_1),
    ("lr", pipeline_2),
    ("rf", pipeline_3),
    ("svc", pipeline_4),
]

In [11]:
stacked_clf = StackingClassifier(
    estimators=estimators, final_estimator=LogisticRegression(),n_jobs=-1
)

In [12]:
stacked_clf.fit(X_train, y_train)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\preprocessing\_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\preprocessing\_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


StackingClassifier(estimators=[('cat',
                                Pipeline(steps=[('preprocessor',
                                                 ColumnTransformer(transformers=[('num',
                                                                                  Pipeline(steps=[('poly',
                                                                                                   PolynomialFeatures()),
                                                                                                  ('select',
                                                                                                   SelectKBest(k=24))]),
                                                                                  ['BLUE',
                                                                                   'GREEN',
                                                                                   'RED',
                                                                                   'NIR',
                                                                                   'SWIR',
                                                                                   'NDVI',
                                                                                   'NDBI',
                                                                                   'REI']),
                                                                                 ('onehot',
                                                                                  OneHotEncoder(dtype=<class 'int'>,
                                                                                                sparse_output=False),
                                                                                  ['NDVI_bin']),
                                                                                 ('ordinal',
                                                                                  OrdinalEncoder(cat...
                                                                                  OneHotEncoder(dtype=<class 'int'>,
                                                                                                sparse_output=False),
                                                                                  ['NDVI_bin']),
                                                                                 ('ordinal',
                                                                                  OrdinalEncoder(categories=[['low_veg',
                                                                                                              'medium_veg',
                                                                                                              'high_veg']],
                                                                                                 dtype=<class 'int'>),
                                                                                  ['NDVI_dis'])])),
                                                ('scale', MinMaxScaler()),
                                                ('dim_reduce',
                                                 LinearDiscriminantAnalysis(n_components=3)),
                                                ('classifier',
                                                 SVC(class_weight={0: 1, 1: 1,
                                                                   2: 10,
                                                                   3: 1}))]))],
                   final_estimator=LogisticRegression(), n_jobs=-1)

In [13]:
# Predict the labels of the test set
y_train_pred = stacked_clf.predict(X_train)
y_test_pred = stacked_clf.predict(X_test)

# Calculate the accuracy of the VotingClassifier
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)}")
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_test_pred)}")

Train Accuracy: (0.9751937984496124, 0.9763094603002871, 0.9751937984496124, 0.9753171760040111)
Train Accuracy: (0.9318181818181818, 0.9325142945949271, 0.9318181818181818, 0.9320257981627175)


In [14]:
# save the model using pickle
# merge the train and test datasets

X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

# train the model using the best hyperparameters and the whole dataset
stacked_clf.fit(X_all, y_all)
with open('trained_models/stacked.pkl', 'wb') as file:
    pickle.dump(stacked_clf, file)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\preprocessing\_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\preprocessing\_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
